In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("students_feature_engineered_v2.csv")

np.random.seed(42)

n_samples = 1000
df_large = df.sample(
    n=n_samples,
    replace=True,
    random_state=42
).reset_index(drop=True)

df_large.to_csv(
    "students_feature_engineered_v2_large.csv",
    index=False
)

print(df_large.shape)

(1000, 7)


In [4]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import ( 
train_test_split, GridSearchCV, cross_val_score) 
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler 
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import ( 
accuracy_score, classification_report, confusion_matrix) 

df = pd.read_csv("students_feature_engineered_v2_large.csv")
X = df.drop(columns=["Pass"]) 
y = df["Pass"] 

X_train, X_test, y_train, y_test = train_test_split( 
X, y, test_size=0.20, random_state=42, stratify=y) 
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows") 


base_pipe = Pipeline([ 
("scaler", StandardScaler()), 
("clf", DecisionTreeClassifier(random_state=42)) 
]) 
base_scores = cross_val_score(base_pipe, X_train, y_train, 
cv=5, scoring="accuracy") 
print(f"Baseline CV: {base_scores.mean():.3f} +/- {base_scores.std():.3f}") 


param_grid = { 
"clf__max_depth": [2, 3, 4, 5], 
"clf__min_samples_split": [2, 5, 10], 
"clf__min_samples_leaf": [1, 2, 4], 
}

grid = GridSearchCV(base_pipe, param_grid, cv=5, 
scoring="accuracy", n_jobs=-1, refit=True) 
grid.fit(X_train, y_train) 
print("Best params:", grid.best_params_) 
print(f"Tuned CV accuracy: {grid.best_score_:.3f}") 


best = grid.best_estimator_ 
y_pred = best.predict(X_test) 
print(f"\nFinal Test Accuracy: {accuracy_score(y_test, y_pred):.3f}") 
print("\nClassification Report:") 
print(classification_report(y_test, y_pred, 
target_names=["Fail","Pass"])) 
print("Confusion Matrix:") 
print(confusion_matrix(y_test, y_pred))


results = X_test.copy() 
results["y_true"] = y_test.values 
results["y_pred"] = y_pred 
results.to_csv("test_predictions.csv", index=False) 
print("Saved: test_predictions.csv")


Train: 800 rows | Test: 200 rows
Baseline CV: 1.000 +/- 0.000
Best params: {'clf__max_depth': 2, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2}
Tuned CV accuracy: 1.000

Final Test Accuracy: 1.000

Classification Report:
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00        68
        Pass       1.00      1.00      1.00       132

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Confusion Matrix:
[[ 68   0]
 [  0 132]]
Saved: test_predictions.csv
